# Context-privileged knowledge distillation (CP-KD) on GPU

**Question:** a reply like *"so true, 100% agree"* is hateful or harmless depending on the comment it answers. Can a fast student that reads **only the reply** learn from a teacher that reads **the parent too**, and can it tell us **when a reply needs its context**?

| Step | What it is |
|---|---|
| C0 | DistilBERT on the reply alone |
| C1 | + KD from a BERT teacher that also reads only the reply (paper-style) |
| C2 | the teacher reads **parent + reply** |
| C3 | + reliability gate |
| C4 | + **needs-context head** = CP-KD (final) |
| U1 | upper bound: the student also reads the parent (needs the thread at inference) |

Datasets download automatically: **rc** = Reddit hate replies with their parent (Yu et al., NAACL 2022), and **ccc** = Civil Comments in Context (Xenos et al.), whose human label flips are the ground truth for "needs context".

Setup is the same as `gpu_ladder.ipynb` (GPU + Internet on, run from the top; finished steps are skipped on re-run). The time estimates are rough guesses: check the seconds per epoch in the logs.

In [ ]:
# 1. Check the GPU
import torch, subprocess
assert torch.cuda.is_available(), "No GPU - switch the runtime/accelerator to GPU first"
print(torch.cuda.get_device_name(0))
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

In [ ]:
# 2. Get the code and the few extra libraries (torch/transformers are already installed on Colab/Kaggle)
import os, sys
REPO = "/content/hate-rkd" if os.path.exists("/content") else "/kaggle/working/hate-rkd"
if not os.path.exists(REPO):
    !git clone -q https://github.com/akashnegetive/hate-rkd.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!pip install -q emoji wordsegment ftfy
!git log --oneline -1

In [ ]:
# 3. Where results and teacher checkpoints go
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/hate-rkd-runs"     # results survive disconnects
    TEACHER_DIR = "/content/teachers"                 # big checkpoints stay on local disk
else:                                                 # Kaggle
    OUT = "/kaggle/working/hate-rkd-runs"
    TEACHER_DIR = "/tmp/teachers"
os.makedirs(OUT, exist_ok=True)
os.environ.update(OUT=OUT, TEACHER_DIR=TEACHER_DIR, CLEAN_TEACHERS="1")
print("results ->", OUT, "| teachers ->", TEACHER_DIR)

In [ ]:
# 3b. Resume (Kaggle): bring back results from an earlier session, so finished steps are skipped.
# Attach them first: right panel -> Add Input -> your earlier notebook version's Output,
# or upload hate-rkd-results.zip as a dataset. Does nothing if nothing is attached.
import glob, shutil
for prev in glob.glob("/kaggle/input/**/hate-rkd-runs", recursive=True):
    shutil.copytree(prev, OUT, dirs_exist_ok=True, ignore=shutil.ignore_patterns("teachers"))
    print("resumed from", prev)
done = sorted(glob.glob(f"{OUT}/results/*__student.json"))
print(len(done), "finished steps found:", [os.path.basename(f)[:-len("__student.json")] for f in done])

## Step A: Reddit, seed 42 (all steps)

In [ ]:
!CLEAN_TEACHERS=0 DATASETS="rc" SEEDS="42" bash scripts/run_context_ladder.sh 2>&1 | grep -E ">>>|\\[skip\\]|epoch|TEACHER|TEST|flag|FAILED|DONE"
!python scripts/make_context_tables.py --runs {OUT} > /dev/null
from IPython.display import Markdown, display
display(Markdown(open(f"{OUT}/context_ladder.md").read()))

## Step B: Civil Comments in Context, seed 42
This is where the needs-context flag is tested against human label flips.

In [ ]:
!DATASETS="ccc" SEEDS="42" bash scripts/run_context_ladder.sh 2>&1 | grep -E ">>>|\\[skip\\]|epoch|TEACHER|TEST|flag|FAILED|DONE"

## Step C: seeds 1 and 2 on both datasets (mean ± std and significance)

In [ ]:
!DATASETS="rc ccc" SEEDS="1 2" bash scripts/run_context_ladder.sh 2>&1 | grep -E ">>>|\\[skip\\]|epoch|TEACHER|TEST|flag|FAILED|DONE"

## Results

In [ ]:
!python scripts/make_context_tables.py --runs {OUT} > /dev/null
display(Markdown(open(f"{OUT}/context_ladder.md").read()))

In [ ]:
!cd {OUT}/.. && zip -qr hate-rkd-context-results.zip {os.path.basename(OUT)} -x "*/teachers/*"
print("zip:", os.path.join(os.path.dirname(OUT), "hate-rkd-context-results.zip"))
if IN_COLAB:
    from google.colab import files
    files.download(os.path.join(os.path.dirname(OUT), "hate-rkd-context-results.zip"))